# AOU-1 — chr22 / nano SMOKE variant (parameterized). Phase M3 / Wave 1+ validation.

Fires `load_qc_cohort` with a single notebook variable `INTERVAL` (default `chr22`; set to a nano sub-interval like `chr22:16000000-18000000` for the Tier 1 cheap fire) against the AoU v8 (or v9 post-migration) controlled-tier WGS MatrixTable. Emits 3 INTERVAL-suffixed checkpointed MTs for live-Hail validation of the m3-W1 Track 4 defensive assertions BEFORE any full-cohort rebuild fires.

**This single notebook now serves BOTH tiers** of the tiered cheap-first validation sequence (see `TIERED-VALIDATION-RUNBOOK.md`):
- **Tier 1 (nano):** `INTERVAL = "chr22:16000000-18000000"` — ~2 Mb, gene-dense, a strict subset of chr22. Keeps the EXACT catastrophe-triggering 2048-partition profile (`naive_coalesce(2048)` + `repartition(2048)` in `aou_ld_panel.py` are FIXED regardless of `interval_filter`) while making per-partition data trivial.
- **Tier 2 (chr22):** `INTERVAL = "chr22"` — the whole chromosome; the first tier that approaches genome-scale memory pressure.

**RIGOR CAVEAT (load-bearing).** The tiers form a memory-pressure gradient. A cheap tier FAILING definitively rules the catastrophe IN (a cheap win). A cheap tier PASSING does NOT exonerate — only the chr22 (Tier 2) / full-genome fire reaches genome-scale memory pressure. **Never label a cheap-tier (nano) pass "validated"; label it "no cheap failure mode reproduced, escalating to the real test."**

**Purpose.** Validate that:
1. `_assert_checkpoint_nonempty(mt, uri, *, phase)` raises loudly on empty contents (or, in the happy case, returns silently with `count_rows() + count_cols() > 0`). **This is the HARD, interval-agnostic catastrophe gate and is UNCHANGED.**
2. The du soft-floor cells (3.5 / 4.5 / 5.5) fire cleanly with an **INTERVAL-scaled** threshold via `_interval_scaled_du_floor(INTERVAL, base_floor_bytes=...)` — so a ~2 Mb nano fire no longer false-positives the old hardcoded 50 MB floor. The du-floor is a DIAGNOSTIC soft-signal; the `count>0` assertion above is the real gate.
3. The auto-resume gate (`_validate_checkpoint_populated`) correctly distinguishes populated MTs from any stub-pattern outputs.
4. The underlying Hail / Dataproc image produces populated MTs at all — if the catastrophe mechanism reproduces, we know to pivot Wave 2 to 1000G AFR.

On ANY Track-4 halt, the du-floor cells call `_capture_catastrophe_forensics(uri, phase=...)` (best-effort, never raises) before re-raising — capturing the `_SUCCESS`-mtime-vs-part-mtimes hypothesis distinguisher (`[[feedback_w1_catastrophe_hypothesis_distinguisher]]`), the MT listing, a `/tmp/hail.log` preserve, and a Spark-REST snapshot.

**Cost.** Tier 1 (nano) shares the Tier 0 64-vCPU cluster (~$1-3). Tier 2 (chr22) ~$35-80 on 24× n1-highmem-16 = 384 vCPU. See the runbook for cluster specs + decision gates. Carter holds every launch / $ trigger.

**Prerequisite.** Gate A (AOU-0.5 mechanism probe) must PASS, and AOU-0 pre-check (`.planning/notebooks/AOU-0-precheck_template.ipynb`) must pass — clone has Track 4 patches, env vars set, distinguisher recorded.

**Outputs** (INTERVAL-suffixed; nano outputs do NOT collide with chr22 outputs):
- `gs://${WORKSPACE_BUCKET}/ld/mt_afr_qc<suffix>.mt`
- `gs://${WORKSPACE_BUCKET}/ld/mt_afr_pca_selfid_qc<suffix>.mt`
- `gs://${WORKSPACE_BUCKET}/ld/mt_eur_qc<suffix>.mt`

where `<suffix>` is `_chr22` for Tier 2 or `_chr22_16000000_18000000` for the Tier 1 nano fire. A `cohort_summary_m3<suffix>.tsv` is written instead of `cohort_summary_m3.tsv`.

**Cross-references:**
- `TIERED-VALIDATION-RUNBOOK.md` — Gate A/B/C decision tree + $ envelopes
- `.planning/notebooks/AOU-0.5-mechanism-probe_template.ipynb` — Tier 0 probe
- `.planning/notebooks/AOU-1_template.ipynb` — the production AOU-1 (do NOT modify)
- `.planning/notebooks/AOU-2-AOU-4-TRACK-4-PATTERN.md` — full-genome Wave 2 target
- `[[feedback_aou_dataproc_pyspark_submit_args]]` — Cell 1a lever
- `[[feedback_aou_cluster_sizing_for_ld_panel]]` — cluster sizing
- `[[feedback_w1_catastrophe_hypothesis_distinguisher]]` — forensics


In [ ]:
# Cell 1a — Force Spark executor resources at the spark-submit boundary.
# CANONICAL PATTERN (per .planning/memory feedback_aou_dataproc_pyspark_submit_args
# baked 2026-05-12): on AoU's Dataproc + YARN cluster, hl.init(spark_conf=dict) is
# silently overridden by the cluster's spark-defaults.conf — the dict path doesn't
# beat YARN's executor-allocation policy. PYSPARK_SUBMIT_ARGS injected BEFORE any
# pyspark/hail import IS honored because it applies at the spark-submit boundary
# (highest Spark conf precedence).
#
# This cell MUST run before any other pyspark/hail import in the notebook.
# Pairs with naive_coalesce(2048) in aou_ld_panel.py:218 (DEC-2026-05-04-01
# v8 partition-explosion OOM remediation; anchor commit 8cc6f64).
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--conf spark.executor.cores=1 "
    "--conf spark.executor.memory=5g "
    "--conf spark.driver.cores=1 "
    "pyspark-shell"
)
print("PYSPARK_SUBMIT_ARGS set:", os.environ["PYSPARK_SUBMIT_ARGS"])

In [ ]:
# Cell 1b — Initialize Hail with spark_conf threading + verify executor.cores=1.
# Calls hl.init directly (NOT through init_hail wrapper) because the wrapper's
# spark_conf path is known broken on AoU YARN; the PYSPARK_SUBMIT_ARGS lever
# from Cell 1a is what actually binds the conf. The spark_conf dict here is
# belt-and-suspenders for portability — preserves the conf-by-dict path for
# environments where it works (local Spark, standalone clusters), while AoU
# binds via the env-var lever.
import sys
sys.path.insert(0, "/home/jupyter/coloc_analysis/src/python")
import hail as hl
hl.init(
    default_reference="GRCh38",
    log="/tmp/hail.log",
    quiet=True,
    spark_conf={
        "spark.executor.cores": "1",
        "spark.executor.memory": "5g",
        "spark.driver.cores": "1",
    },
)
# Pull cohort helpers AFTER Hail backend is up (so any aou_ld_panel-side
# Hail-dependent imports succeed):
from aou_ld_panel import (load_qc_cohort, ANCESTRY_FIELD, KING_KINSHIP_THRESHOLD, _qc_checkpoint_uri,
                          _interval_scaled_du_floor, _capture_catastrophe_forensics)

# Verify the patches are live (cores=1 confirms PYSPARK_SUBMIT_ARGS bound;
# _qc_checkpoint_uri import confirms commit 36e8062 is in the AoU clone):
sc_conf = hl.spark_context().getConf()
cores = sc_conf.get('spark.executor.cores')
assert cores == '1', (
    f"PYSPARK_SUBMIT_ARGS lever did not bind — got cores={cores}, expected '1'. "
    f"DO NOT proceed to Cell 3+ — the v8 partition-explosion OOM config is NOT live. "
    f"Action: Kernel menu → Restart Kernel; then re-fire Cell 1a + Cell 1b."
)
print("=== HAIL INIT ===")
print(f"  Hail version          : {hl.__version__}")
print(f"  spark.executor.cores  : {cores}  OK")
print(f"  spark.executor.memory : {sc_conf.get('spark.executor.memory')}")
print(f"  spark.driver.cores    : {sc_conf.get('spark.driver.cores')}")
print(f"  spark.master          : {sc_conf.get('spark.master')}")
print()
print("=== ENV ===")
print(f"  WORKSPACE_BUCKET = {os.environ['WORKSPACE_BUCKET']}")
print(f"  GOOGLE_PROJECT   = {os.environ['GOOGLE_PROJECT']}")
print(f"  WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH = {os.environ['WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH']}")
print()
print("=== PATCH VERIFICATION (commit 36e8062 — m3-W1-checkpoint-suffix; quick 260512-jd9) ===")
print(f"  AFR primary URI     : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False)}")
print(f"  AFR sensitivity URI : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True)}")
print(f"  EUR parity URI      : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False)}")

In [ ]:
# Cell 1c — SINGLE tier selector. This is the ONLY knob that switches the
# notebook between Tier 1 (nano) and Tier 2 (chr22). It is threaded into all
# THREE load_qc_cohort calls (as the interval-filter arg), all three du-floor
# cells, the output suffixes, and the cohort-summary table.
#
#   Tier 2 (chr22, default)  : leave INTERVAL at its default below
#   Tier 1 (nano, ~2 Mb)     : set INTERVAL to a chr22 sub-span (start-end)
#
# RIGOR: a nano (Tier 1) PASS is "no cheap failure mode reproduced", NOT
# "validated" — only Tier 2 / full-genome reaches genome-scale memory pressure.
INTERVAL = "chr22"  # set to "chr22:16000000-18000000" for the Tier 1 nano fire (~2 Mb, gene-dense, strict subset of chr22)

# Output-suffix derivation (matches the _intermediate_checkpoint_uri convention):
#   'chr22'                        -> '_chr22'
#   'chr22:16000000-18000000'      -> '_chr22_16000000_18000000'
_suffix = "_" + INTERVAL.replace(":", "_").replace("-", "_")

# INTERVAL-scaled du soft-floor (Task-1 helper). The base is the whole-
# chromosome / full-genome floor (50 MB, written as arithmetic so the bare
# literal lives only in the helper's caller policy, not as a magic number);
# a span-bounded nano interval scales it DOWN proportionally so a ~2 Mb fire no
# longer false-positives. The count_rows>0/count_cols>0 assertion inside
# load_qc_cohort is the HARD catastrophe gate; this du-floor is a SOFT signal.
_BASE_DU_FLOOR_BYTES = 50 * 10**6  # 50 MB whole-chromosome / full-genome base floor
_DU_FLOOR_BYTES = _interval_scaled_du_floor(INTERVAL, base_floor_bytes=_BASE_DU_FLOOR_BYTES)
print(f"INTERVAL = {INTERVAL!r}  ->  suffix = {_suffix!r}  ->  du soft-floor = {_DU_FLOOR_BYTES:,} bytes")


In [ ]:
# Cell 3 — Primary AFR cohort (D-M3-07 PCA-primary) — chr22 SMOKE
mt_afr = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=False,
    interval_filter=INTERVAL,  # tier selector (DESIGN §3.5 path-isolated execution)
)
n_afr = mt_afr.count_cols()
n_var_afr = mt_afr.count_rows()
print(f"AFR PCA cohort ({INTERVAL}): {n_afr} samples, {n_var_afr} variants")
# Already checkpointed to gs://${WORKSPACE_BUCKET}/ld/mt_afr_qc_chr22.mt by load_qc_cohort()


In [ ]:
# Cell 3.5 — Mandatory post-write bucket-contents validation (chr22 SMOKE variant).
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard, INTERVAL-scaled.
#
# The du-floor is a DIAGNOSTIC SOFT-SIGNAL scaled to the INTERVAL span (Task-1
# _interval_scaled_du_floor): chr22/full-genome -> full 50 MB base; a ~2 Mb nano
# interval -> a proportionally small few-MB floor so it does NOT false-positive a
# legitimately small nano cohort. The REAL catastrophe gate is count_rows>0/
# count_cols>0 in _assert_checkpoint_nonempty inside load_qc_cohort (UNCHANGED).
# (The footer-stub catastrophe state is ~70 KiB total entries — well below
# any scaled floor — so the soft-floor still flags it on any tier.)
#
# Cell URI uses the INTERVAL-derived suffix (_suffix from Cell 1c) per
# the _intermediate_checkpoint_uri / _qc_checkpoint_uri convention. Note that
# _qc_checkpoint_uri itself does NOT honor interval_filter (that's intermediate-
# only); we suffix manually so nano outputs do NOT collide with chr22 outputs.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
# - [[feedback_w1_catastrophe_hypothesis_distinguisher]]
#
# Cohort: Primary AFR chr22 smoke
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False) + _suffix
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/entries/parts/'
try:
    _r = subprocess.run(
        ['gsutil', 'du', '-s', _entries_dir],
        capture_output=True, text=True,
    )
    assert _r.returncode == 0, (
        f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
        f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
        f'the m3-W1 empty-MT catastrophe signature reproducing under this tier. '
        f'HALT — do not proceed to next cell. Notify Abby Doyle (Zendesk #57144) '
        f'and pivot Wave 2 to 1000G AFR substrate.'
    )
    _size_bytes = int(_r.stdout.split()[0])
    assert _size_bytes > _MIN_BYTES, (
        f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
        f'(< {_MIN_BYTES:,} bytes INTERVAL-scaled soft-floor) — empty-MT catastrophe '
        f'regression guard. This tier write produced a sub-floor MT; do NOT proceed. '
        f'See .planning/debug/m3-W1-empty-mt-catastrophe.md and the TIERED-VALIDATION-RUNBOOK.md '
        f'failure-mode matrix.'
    )
    print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**6:.1f} MB at entries/entries/parts/)')
except Exception:
    # Best-effort forensic capture BEFORE re-raising — never weakens the halt.
    # _capture_catastrophe_forensics is defensive (never raises); the re-raise
    # below is what halts the cell loudly.
    _capture_catastrophe_forensics(_ckpt_uri, phase='afr')
    raise


In [ ]:
# Cell 4 — AFR sensitivity cohort (D-M3-07 self-report Black/African American sensitivity) — chr22 SMOKE
mt_afr_selfid = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=True,
    interval_filter=INTERVAL,
)
n_afr_selfid = mt_afr_selfid.count_cols()
print(f"AFR PCA + self-id Black/AA cohort ({INTERVAL}): {n_afr_selfid} samples (subset of AFR PCA chr22 cohort)")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_afr_pca_selfid_qc_chr22.mt


In [ ]:
# Cell 4.5 — Mandatory post-write bucket-contents validation (chr22 SMOKE variant).
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard, INTERVAL-scaled.
#
# The du-floor is a DIAGNOSTIC SOFT-SIGNAL scaled to the INTERVAL span (Task-1
# _interval_scaled_du_floor): chr22/full-genome -> full 50 MB base; a ~2 Mb nano
# interval -> a proportionally small few-MB floor so it does NOT false-positive a
# legitimately small nano cohort. The REAL catastrophe gate is count_rows>0/
# count_cols>0 in _assert_checkpoint_nonempty inside load_qc_cohort (UNCHANGED).
# (The footer-stub catastrophe state is ~70 KiB total entries — well below
# any scaled floor — so the soft-floor still flags it on any tier.)
#
# Cell URI uses the INTERVAL-derived suffix (_suffix from Cell 1c) per
# the _intermediate_checkpoint_uri / _qc_checkpoint_uri convention. Note that
# _qc_checkpoint_uri itself does NOT honor interval_filter (that's intermediate-
# only); we suffix manually so nano outputs do NOT collide with chr22 outputs.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
# - [[feedback_w1_catastrophe_hypothesis_distinguisher]]
#
# Cohort: AFR sensitivity chr22 smoke
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True) + _suffix
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/entries/parts/'
try:
    _r = subprocess.run(
        ['gsutil', 'du', '-s', _entries_dir],
        capture_output=True, text=True,
    )
    assert _r.returncode == 0, (
        f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
        f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
        f'the m3-W1 empty-MT catastrophe signature reproducing under this tier. '
        f'HALT — do not proceed to next cell. Notify Abby Doyle (Zendesk #57144) '
        f'and pivot Wave 2 to 1000G AFR substrate.'
    )
    _size_bytes = int(_r.stdout.split()[0])
    assert _size_bytes > _MIN_BYTES, (
        f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
        f'(< {_MIN_BYTES:,} bytes INTERVAL-scaled soft-floor) — empty-MT catastrophe '
        f'regression guard. This tier write produced a sub-floor MT; do NOT proceed. '
        f'See .planning/debug/m3-W1-empty-mt-catastrophe.md and the TIERED-VALIDATION-RUNBOOK.md '
        f'failure-mode matrix.'
    )
    print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**6:.1f} MB at entries/entries/parts/)')
except Exception:
    # Best-effort forensic capture BEFORE re-raising — never weakens the halt.
    # _capture_catastrophe_forensics is defensive (never raises); the re-raise
    # below is what halts the cell loudly.
    _capture_catastrophe_forensics(_ckpt_uri, phase='afr_sens')
    raise


In [ ]:
# Cell 5 — EUR parity cohort (D-M3-01) — chr22 SMOKE
mt_eur = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="eur",
    sensitivity=False,
    interval_filter=INTERVAL,
)
n_eur = mt_eur.count_cols()
print(f"EUR PCA cohort ({INTERVAL}): {n_eur} samples")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_eur_qc_chr22.mt


In [ ]:
# Cell 5.5 — Mandatory post-write bucket-contents validation (chr22 SMOKE variant).
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard, INTERVAL-scaled.
#
# The du-floor is a DIAGNOSTIC SOFT-SIGNAL scaled to the INTERVAL span (Task-1
# _interval_scaled_du_floor): chr22/full-genome -> full 50 MB base; a ~2 Mb nano
# interval -> a proportionally small few-MB floor so it does NOT false-positive a
# legitimately small nano cohort. The REAL catastrophe gate is count_rows>0/
# count_cols>0 in _assert_checkpoint_nonempty inside load_qc_cohort (UNCHANGED).
# (The footer-stub catastrophe state is ~70 KiB total entries — well below
# any scaled floor — so the soft-floor still flags it on any tier.)
#
# Cell URI uses the INTERVAL-derived suffix (_suffix from Cell 1c) per
# the _intermediate_checkpoint_uri / _qc_checkpoint_uri convention. Note that
# _qc_checkpoint_uri itself does NOT honor interval_filter (that's intermediate-
# only); we suffix manually so nano outputs do NOT collide with chr22 outputs.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
# - [[feedback_w1_catastrophe_hypothesis_distinguisher]]
#
# Cohort: EUR parity chr22 smoke
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False) + _suffix
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/entries/parts/'
try:
    _r = subprocess.run(
        ['gsutil', 'du', '-s', _entries_dir],
        capture_output=True, text=True,
    )
    assert _r.returncode == 0, (
        f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
        f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
        f'the m3-W1 empty-MT catastrophe signature reproducing under this tier. '
        f'HALT — do not proceed to next cell. Notify Abby Doyle (Zendesk #57144) '
        f'and pivot Wave 2 to 1000G AFR substrate.'
    )
    _size_bytes = int(_r.stdout.split()[0])
    assert _size_bytes > _MIN_BYTES, (
        f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
        f'(< {_MIN_BYTES:,} bytes INTERVAL-scaled soft-floor) — empty-MT catastrophe '
        f'regression guard. This tier write produced a sub-floor MT; do NOT proceed. '
        f'See .planning/debug/m3-W1-empty-mt-catastrophe.md and the TIERED-VALIDATION-RUNBOOK.md '
        f'failure-mode matrix.'
    )
    print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**6:.1f} MB at entries/entries/parts/)')
except Exception:
    # Best-effort forensic capture BEFORE re-raising — never weakens the halt.
    # _capture_catastrophe_forensics is defensive (never raises); the re-raise
    # below is what halts the cell loudly.
    _capture_catastrophe_forensics(_ckpt_uri, phase='eur')
    raise


In [ ]:
# Cell 6 — Disjoint-cohort sanity check (RESEARCH O5)
afr_samples = mt_afr.s.collect()
eur_samples = mt_eur.s.collect()
overlap = set(afr_samples) & set(eur_samples)
assert len(overlap) == 0, f"AFR and EUR cohorts overlap by {len(overlap)} samples; investigate!"
print(f"OK: AFR and EUR cohorts disjoint ({len(afr_samples)} + {len(eur_samples)} samples)")

In [ ]:
# Cell 7 — Cohort-summary table for the chr22 smoke validation
import pandas as pd
cohort_summary = pd.DataFrame({
    "cohort": [f"AFR_pca{_suffix}", f"AFR_pca_selfid{_suffix}", f"EUR_pca{_suffix}"],
    "n_samples": [n_afr, n_afr_selfid, n_eur],
    "n_variants": [n_var_afr, mt_afr_selfid.count_rows(), mt_eur.count_rows()],
    "kinship_threshold": [KING_KINSHIP_THRESHOLD] * 3,
    "ancestry_field": [ANCESTRY_FIELD] * 3,
    "checkpoint_path": [
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False) + _suffix,
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True) + _suffix,
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False) + _suffix,
    ],
    "interval_filter": [INTERVAL] * 3,
})
cohort_summary.to_csv("cohort_summary_m3" + _suffix + ".tsv", sep="\t", index=False)
print(cohort_summary)
print()
print(f"{INTERVAL} smoke complete. If all 3 cohort-summary rows show non-zero n_samples AND n_variants,")
print("the HARD count>0 gate held for this tier. RIGOR: a nano (Tier 1) pass is")
print("'no cheap failure mode reproduced, escalating to the real test' — NOT 'validated'.")
print("Only the chr22 (Tier 2) / full-genome fire reaches genome-scale memory pressure")
print("(assuming Track 1 credit recovery is also resolved). If any row shows 0, the catastrophe")
print("mechanism reproduces under the current platform — pivot Wave 2 to 1000G AFR.")


## Tier output: 3 INTERVAL-suffixed checkpointed MTs in the workspace bucket + `cohort_summary_m3<suffix>.tsv` on env local disk.

Mirror `cohort_summary_m3<suffix>.tsv` to NCSU GPFS at `.planning/phases/m3-aou-afr-ld-panel-build/` after the fire (commit + push).

**RIGOR REMINDER.** If this was the **Tier 1 nano** fire (`INTERVAL = "chr22:16000000-18000000"`), a PASS is **"no cheap failure mode reproduced, escalating to the real test"** — NOT "validated". Tear down the cluster and proceed to the **Tier 2 chr22** fire (leave `INTERVAL` at its `chr22` default) per the Gate B/C decision tree in `TIERED-VALIDATION-RUNBOOK.md`. Only the Tier 2 / full-genome fire reaches genome-scale memory pressure.

**Next step if a tier passes (Tier 2 / chr22):** plan the full-cohort rebuild on the same Dataproc preset (ref `.planning/notebooks/AOU-2-AOU-4-TRACK-4-PATTERN.md`). The Track 4 assertions in `load_qc_cohort` will catch any silent empty-MT recurrence.

**Next step if any tier fails (empty / sub-floor):** the catastrophe mechanism reproduces under the current platform — pivot Wave 2 to the 1000G AFR substrate (free, NCSU-side, ~11 candidate-locus `.rds` files already at `data/processed/ld_reference/AFR/`). Document the Wave 2 deviation in the OSF amendment trail. Inspect the `_forensics/<phase>_capture.json` the du-floor cell wrote for the `_SUCCESS`-mtime-vs-part-mtimes hypothesis flag.
